# Testar Customer Churn Predictor API no Google Colab

Este notebook te permite testar a API sem precisar instalar nada na sua máquina.

**Passo 0:** suba o arquivo `customer-churn-predictor-api.zip` para o Colab, OU (melhor) suba o projeto pro GitHub primeiro e clone o repositório aqui.

Depois disso, você tem duas formas de testar:
- **Opção A — Teste interno rápido**: valida a lógica da API sem gerar uma URL pública (mais simples, roda tudo dentro do Colab).
- **Opção B — API pública com ngrok**: expõe a API numa URL real que você pode acessar do navegador, Postman, etc.


## 1. Instalar dependências

In [ ]:
!pip install -q fastapi uvicorn scikit-learn pandas numpy joblib pydantic httpx nest-asyncio pyngrok

## 2. Obter o projeto

Escolha **UMA** das duas opções abaixo.

### Opção 2A — Clonar do GitHub (recomendado, depois de você já ter subido o repositório)

In [ ]:

!git clone https://github.com/PedroLucasBarbosa/customer-churn-predictor-api.git
%cd customer-churn-predictor-api

Cloning into 'customer-churn-predictor-api'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 30 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 4.74 MiB | 6.29 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/customer-churn-predictor-api


### Opção 2B — Fazer upload manual do .zip (se ainda não subiu pro GitHub)

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecione o customer-churn-predictor-api.zip

import zipfile, os
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

%cd customer-churn-predictor-api

## 3. Gerar dados e treinar o modelo (caso o .joblib não esteja incluso)

In [ ]:
!python data/generate_data.py
!python src/train.py

Dataset generated: data/telco_churn.csv
Shape: (5000, 21)
Churn rate: 26.16%
logistic_regression: {'accuracy': 0.732, 'f1_score': 0.589, 'roc_auc': np.float64(0.8116)}
random_forest: {'accuracy': 0.771, 'f1_score': 0.5918, 'roc_auc': np.float64(0.8123)}

Best model: random_forest -> {'accuracy': 0.771, 'f1_score': 0.5918, 'roc_auc': np.float64(0.8123)}

Model saved to models/churn_model.joblib
Metadata saved to models/model_metadata.json


## Opção A — Teste rápido e interno (sem URL pública)

Usa o `TestClient` do FastAPI, que simula requisições HTTP sem precisar subir um servidor de verdade. Ideal para validar rapidinho se tudo está funcionando.

In [ ]:
from fastapi.testclient import TestClient
from src.main import app

with TestClient(app) as client:
    print(client.get('/health').json())

    payload = {
        "gender": "Female", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No",
        "tenure": 5, "PhoneService": "Yes", "MultipleLines": "No",
        "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
        "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
        "StreamingMovies": "Yes", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
        "PaymentMethod": "Electronic check", "MonthlyCharges": 85.5, "TotalCharges": 427.5
    }

    r1 = client.post('/predict', json=payload)
    print(r1.status_code, r1.json())

    r2 = client.post('/predict', json=payload)
    print(r2.status_code, r2.json())

{'status': 'ok', 'model_loaded': True, 'model_version': '1.0.0'}
200 {'churn_prediction': 'Yes', 'churn_probability': 0.8836, 'risk_level': 'High'}
200 {'churn_prediction': 'Yes', 'churn_probability': 0.8836, 'risk_level': 'High'}


## Opção B — API pública com ngrok

Sobe a API de verdade dentro do Colab e gera uma URL pública temporária, que você pode abrir no navegador (`/docs`), testar no Postman, ou chamar de qualquer lugar.

**Você vai precisar de um token gratuito do ngrok:**
1. Crie uma conta em https://dashboard.ngrok.com/signup
2. Copie seu authtoken em https://dashboard.ngrok.com/get-started/your-authtoken
3. Cole no lugar de `SEU_TOKEN_AQUI` abaixo

In [ ]:
from pyngrok import ngrok, conf

conf.get_default().auth_token = "SEU_TOKEN_AQUI"

import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print("API pública disponível em:", public_url)
print("Documentação interativa:", str(public_url) + "/docs")

def run():
    uvicorn.run("src.main:app", host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()

Abra o link de `/docs` impresso acima em outra aba — ele carrega o Swagger UI, onde dá pra testar o endpoint `/predict` direto pelo navegador, sem precisar escrever código.

Ou teste programaticamente com `requests`, na célula abaixo:

In [ ]:
import requests
import time

time.sleep(2)  # dá um tempinho pro servidor subir

payload = {
    "gender": "Female", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No",
    "tenure": 5, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
    "StreamingMovies": "Yes", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 85.5, "TotalCharges": 427.5
}

r = requests.post(f"{public_url}/predict", json=payload)
print(r.status_code)
print(r.json())

## 4. Encerrar (opcional)

Quando terminar os testes, feche o túnel do ngrok:

In [ ]:
ngrok.disconnect(public_url)